In [0]:
import requests
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.sql.functions import current_timestamp, lit

file_path = "/Volumes/workspace/stock_data/api_keys/fmp_api_key.txt"

with open(file_path, "r") as f:
    api_key = f.read().strip()

ticker_df = spark.sql("SELECT symbol FROM workspace.stock_data.company_profiles_silver")

tickers = [row['symbol'] for row in ticker_df.collect()]

def get_income_statement_raw(ticker, api_key, period, limit):
    url = "https://financialmodelingprep.com/stable/income-statement"
    params = {
        "symbol": ticker,
        "apikey": api_key,
        "period": period,
        "limit": limit
    }
    response = requests.get(url, params=params)
    return (ticker, response.text, "Income Statement", period, None)

def get_balance_sheet(ticker, api_key, period, limit):
    url = f"https://financialmodelingprep.com/stable/balance-sheet-statement"
    params = {
        'symbol' : f'{ticker}',
        'apikey' : f'{api_key}',
        'period' : f'{period}',
        'limit' : f'{limit}'
    }
    response = requests.get(url, params=params)
    return (ticker, response.text, "Balance Sheet", period, None)

bronze_schema = StructType([
    StructField("ticker", StringType(), True),
    StructField("raw_json", StringType(), True),
    StructField("source", StringType(), True),
    StructField("period_type", StringType(), True),
    StructField("ingest_timestamp", TimestampType(), True)
])

income_statement_data = []

balance_sheet_data = []

# quarterly
income_statement_data += [
    get_income_statement_raw(ticker, api_key, "quarter", 40)
    for ticker in tickers
]

balance_sheet_data += [
    get_balance_sheet(ticker, api_key, "quarter", 40)
    for ticker in tickers
]

# annual
income_statement_data += [
    get_income_statement_raw(ticker, api_key, "annual", 10)
    for ticker in tickers
]

balance_sheet_data += [
    get_balance_sheet(ticker, api_key, "annual", 10)
    for ticker in tickers
]

# create dataframes
bronze_is_df = (
    spark.createDataFrame(income_statement_data, schema=bronze_schema)
         .withColumn("ingest_timestamp", current_timestamp())
)

bronze_bs_df = (
    spark.createDataFrame(balance_sheet_data, schema=bronze_schema)
         .withColumn("ingest_timestamp", current_timestamp())
)

# write to tables
bronze_is_df.write.format("delta").mode("overwrite").saveAsTable('workspace.stock_data.income_statements_bronze')

bronze_bs_df.write.format("delta").mode("overwrite").saveAsTable('workspace.stock_data.balance_sheets_bronze')

In [0]:
%sql
SELECT *
FROM workspace.stock_data.income_statements_bronze

In [0]:
%sql
SELECT *
FROM workspace.stock_data.balance_sheets_bronze